# Laboratorio 4 — Informe final de pruebas y resultados

Este notebook consolida la evaluación del asistente de compras por voz: **14 escenarios de texto y 6 de audio**, restricciones de presupuesto y stock, consentimiento, ASR, latencias y calidad TTS opcional.

> Por defecto, `Run All` revisa la evidencia guardada sin cargar modelos ni repetir pruebas. Para una ejecución nueva, activa las banderas de la sección 1.


## 1. Configuración de la ejecución


In [1]:
from pathlib import Path
import importlib.metadata
import json
import sys
import time
from datetime import datetime

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'src' / 'lab4_asistente').is_dir():
    ROOT = next((p for p in ROOT.parents if (p / 'src' / 'lab4_asistente').is_dir()), ROOT)
sys.path.insert(0, str(ROOT / 'src'))

from lab4_asistente import config as C
from lab4_asistente.escenarios import ESCENARIOS, ejecutar
from lab4_asistente.evaluacion import (
    resumen_escenarios, tabla_aciertos_fallos, tabla_asr,
    tabla_calidad_tts, tabla_escenarios, tabla_latencias,
)
from lab4_asistente.runtime import cargar_modelos_voz, cargar_sistema
from lab4_asistente.voz import Semejanza

# Modo seguro para Run All: revisar resultados ya guardados.
EJECUTAR_TEXTO = False
EJECUTAR_VOZ = False
EJECUTAR_REPLAY_TEXTO = False
SINTETIZAR_TTS = False
EVALUAR_TTS = False
GUARDAR_INFORME = True

PERSONA = 'diego'             # audios del comprador evaluado
VOICE_ID = 'ana'              # voz sintetizada del asistente

print('Raíz del proyecto:', ROOT)
print('Persona evaluada:', PERSONA, '| Voz del asistente:', VOICE_ID)
print('Nueva ejecución:', EJECUTAR_TEXTO or EJECUTAR_VOZ or EJECUTAR_REPLAY_TEXTO)


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz
Persona evaluada: diego | Voz del asistente: ana
Nueva ejecución: False


## 2. Fuente de evidencia y carga condicional

Una ejecución nueva carga Qwen una sola vez. Whisper/XTTS solo se cargan si se activa voz o la evaluación TTS. En modo informe, los resultados se reconstruyen desde los `registro.json`.


In [2]:
def cargar_resultado_guardado(carpeta):
    registro = Path(carpeta) / 'registro.json'
    estado_ruta = Path(carpeta) / 'estado_final.json'
    if not registro.is_file():
        raise FileNotFoundError(f'No existe evidencia: {registro}')
    datos = json.loads(registro.read_text(encoding='utf-8'))
    estado = json.loads(estado_ruta.read_text(encoding='utf-8')) if estado_ruta.is_file() else {}
    escenario = datos['escenario']
    ver = datos['verificacion']
    return {
        'id': escenario['id'], 'modo': datos['modo'], 'tipo': escenario['tipo'],
        'turnos': len(datos.get('turnos', [])), 'completado': ver['ok'],
        'fallos': '; '.join(ver.get('fallos', [])),
        'incumplimientos': ','.join(ver.get('incumplimientos', [])),
        'pedidos': len(estado.get('pedidos', [])), 'registro': datos.get('turnos', []),
    }

NECESITA_SISTEMA = EJECUTAR_TEXTO or EJECUTAR_VOZ or EJECUTAR_REPLAY_TEXTO
COMP = SIS = None
CARGAS_INICIALES = {}
if NECESITA_SISTEMA:
    COMP = cargar_sistema(cargar_llm=True)
    SIS = COMP['sistema']
    CARGAS_INICIALES.update(COMP['cargas'])
    print('Sistema cargado:', COMP['dispositivos'])
else:
    print('Modo informe: no se cargó el LLM.')


Modo informe: no se cargó el LLM.


## 3. Pruebas de texto — 14 escenarios


In [3]:
ESCENARIOS_TEXTO = [e for e in ESCENARIOS if e['modo'] == 'texto']
RES_TEXTO = []
if EJECUTAR_TEXTO:
    for e in ESCENARIOS_TEXTO:
        RES_TEXTO.append(ejecutar(
            e, SIS, 'texto', C.RUTA_ESCENARIOS / 'texto' / e['id'], progreso=True,
        ))
else:
    RES_TEXTO = [
        cargar_resultado_guardado(C.RUTA_ESCENARIOS / 'texto' / e['id'])
        for e in ESCENARIOS_TEXTO
    ]

MATRIZ_TEXTO = tabla_escenarios(RES_TEXTO)
display(MATRIZ_TEXTO)
print('Resumen texto:', resumen_escenarios(RES_TEXTO))


,id,modo,tipo,completado,fallos,incumplimientos,pedidos
0,T07,texto,compra simple con presupuesto,True,,,1
1,T08,texto,varios productos con presupuesto total,True,,,1
2,T09,texto,presupuesto insuficiente,True,,,0
3,T10,texto,atributo no verificable,True,,,0
4,T11,texto,agotado sin solución,True,,,0
5,T12,texto,cantidad mayor que el stock,True,,,1
6,T13,texto,compra con cotización y consentimiento posterior,True,,,1
7,T14,texto,quitar un producto del carrito,True,,,1
8,T15,texto,cambio de preferencia (color),True,,,1
9,T16,texto,comparar alternativas,True,,,0


Resumen texto: {'total': 14, 'completados': 14, 'porcentaje_completado': 100.0, 'incumplimientos': 0}


## 4. Pruebas de voz — 6 escenarios

La persona se fija en la configuración para no mezclar ejecuciones históricas. `SINTETIZAR_TTS=False` valida ASR, agente y compra sin generar WAV de respuesta.


In [4]:
ESCENARIOS_VOZ = [e for e in ESCENARIOS if e['modo'] == 'voz']
RUTA_AUDIOS = ROOT / 'data' / 'scenario_audio' / PERSONA
RUTA_VOZ_RESULTADOS = C.RUTA_ESCENARIOS / 'voz_audio' / PERSONA
ASR = TTS = VOZ = None

if EJECUTAR_VOZ or EVALUAR_TTS:
    VOZ = C.rutas_voz(VOICE_ID)
    ASR, TTS, cargas_voz = cargar_modelos_voz(VOICE_ID)
    CARGAS_INICIALES.update(cargas_voz)

FALTAN = [
    t['audio'] for e in ESCENARIOS_VOZ for t in e['turnos']
    if not (RUTA_AUDIOS / t['audio']).is_file()
]
print('Audios requeridos:', sum(len(e['turnos']) for e in ESCENARIOS_VOZ), '| faltantes:', len(FALTAN))

RES_VOZ = []
if EJECUTAR_VOZ:
    assert not FALTAN, f'Faltan audios en {RUTA_AUDIOS}: {FALTAN}'
    for e in ESCENARIOS_VOZ:
        RES_VOZ.append(ejecutar(
            e, SIS, 'voz', RUTA_VOZ_RESULTADOS / e['id'],
            ruta_audios=RUTA_AUDIOS, asr=ASR,
            tts=TTS if SINTETIZAR_TTS else None,
            sintetizar=SINTETIZAR_TTS, progreso=True,
        ))
else:
    RES_VOZ = [
        cargar_resultado_guardado(RUTA_VOZ_RESULTADOS / e['id'])
        for e in ESCENARIOS_VOZ
    ]

MATRIZ_VOZ = tabla_escenarios(RES_VOZ)
display(MATRIZ_VOZ)
print('Resumen voz:', resumen_escenarios(RES_VOZ))


Audios requeridos: 23 | faltantes: 0


,id,modo,tipo,completado,fallos,incumplimientos,pedidos
0,E01,voz,compra que cumple restricciones,True,,,1
1,E02,voz,solicitud ambigua con aclaración,True,,,1
2,E03,voz,cambio de cantidad conservando restricciones,True,,,1
3,E04,voz,producto agotado sin alternativa disponible,True,,,0
4,E05,voz,corrección de transcripción antes de comprar,True,,,1
5,E06,voz,cambio de stock entre cotización y confirmación,True,,,1


Resumen voz: {'total': 6, 'completados': 6, 'porcentaje_completado': 100.0, 'incumplimientos': 0}


## 5. Resultado consolidado y cumplimiento


In [5]:
TODOS = RES_TEXTO + RES_VOZ
MATRIZ = tabla_escenarios(TODOS)
RESUMEN = resumen_escenarios(TODOS)

display(MATRIZ[['id', 'modo', 'tipo', 'completado', 'fallos', 'incumplimientos', 'pedidos']])
print('Resumen consolidado:', RESUMEN)
print('Aprobados:', int(MATRIZ['completado'].sum()), '/', len(MATRIZ))
print('Incumplimientos:', sum(bool(x) for x in MATRIZ['incumplimientos']))

FALLIDOS = MATRIZ.loc[~MATRIZ['completado']]
if len(FALLIDOS):
    display(FALLIDOS[['id', 'modo', 'fallos', 'incumplimientos']])
else:
    print('Resultado final: todos los escenarios fueron aprobados.')


,id,modo,tipo,completado,fallos,incumplimientos,pedidos
0,T07,texto,compra simple con presupuesto,True,,,1
1,T08,texto,varios productos con presupuesto total,True,,,1
2,T09,texto,presupuesto insuficiente,True,,,0
3,T10,texto,atributo no verificable,True,,,0
4,T11,texto,agotado sin solución,True,,,0
5,T12,texto,cantidad mayor que el stock,True,,,1
6,T13,texto,compra con cotización y consentimiento posterior,True,,,1
7,T14,texto,quitar un producto del carrito,True,,,1
8,T15,texto,cambio de preferencia (color),True,,,1
9,T16,texto,comparar alternativas,True,,,0


Resumen consolidado: {'total': 20, 'completados': 20, 'porcentaje_completado': 100.0, 'incumplimientos': 0}
Aprobados: 20 / 20
Incumplimientos: 0
Resultado final: todos los escenarios fueron aprobados.


## 6. ASR: WER, CER y entidades críticas


In [6]:
ASR_METRICAS = tabla_asr(RES_VOZ)
display(ASR_METRICAS)
if len(ASR_METRICAS):
    print('WER medio:', ASR_METRICAS.WER.mean())
    print('CER medio:', ASR_METRICAS.CER.mean())
    print('Errores de cantidades/importes/productos:',
          ASR_METRICAS.err_cantidades.sum(),
          ASR_METRICAS.err_importes.sum(),
          ASR_METRICAS.err_productos.sum())


,escenario,turno,referencia,transcripcion,WER,CER,err_cantidades,err_importes,err_productos,n_cantidades,n_importes,n_productos
0,E01,1,"Quiero comprar dos paquetes de pilas AA, mi pr...",Quiero comprar dos paquetes de pilas AA. Mi pr...,0.153846,0.135135,0,0,0,1,1,2
1,E01,2,Agrega la primera opción y prepárame la cotiza...,Agrega la primera opción y prepárame la cotiza...,0.000000,0.000000,0,0,0,0,0,0
2,E01,3,"Sí, confirmo el pedido.","Sí, confirmo el pedido.",0.000000,0.000000,0,0,0,0,0,0
3,E02,1,Quiero comprar una funda.,Quiero comprar una fonda.,0.250000,0.041667,0,0,1,1,0,1
4,E02,2,"Es para un iPhone doce, de color negro.","Es para un iPhone 12, de color negro.",0.125000,0.108108,0,0,0,2,0,3
5,E02,3,Agrega la primera y dime el total.,Agrega la primera y dime el total.,0.000000,0.000000,0,0,0,0,0,1
6,E02,4,"Sí, confirmo.","Sí, confirmo.",0.000000,0.000000,0,0,0,0,0,0
7,E03,1,"Necesito tres botellas de agua para niños, con...","Necesito tres botellas de agua para los niños,...",0.133333,0.146341,0,0,0,2,1,3
8,E03,2,Agrega la más barata.,agrega la más barata,0.000000,0.000000,0,0,0,0,0,0
9,E03,3,"Mejor que sean dos en lugar de tres, y prepára...",Mejor que sean dos en lugar de tres. Y prepára...,0.000000,0.000000,0,0,0,2,0,1


WER medio: 0.13467782217782218
CER medio: 0.08189760726348035
Errores de cantidades/importes/productos: 0 0 6


## 7. Comparación con transcripción correcta (opcional)


In [7]:
RUTA_REPLAY = C.RUTA_ESCENARIOS / 'voz_transcripcion_correcta' / PERSONA
RES_VOZ_TEXTO = []
if EJECUTAR_REPLAY_TEXTO:
    for e in ESCENARIOS_VOZ:
        RES_VOZ_TEXTO.append(ejecutar(
            e, SIS, 'texto', RUTA_REPLAY / e['id'], progreso=True,
        ))
else:
    for e in ESCENARIOS_VOZ:
        if (RUTA_REPLAY / e['id'] / 'registro.json').is_file():
            RES_VOZ_TEXTO.append(cargar_resultado_guardado(RUTA_REPLAY / e['id']))

replay_por_id = {r['id']: r for r in RES_VOZ_TEXTO}
COMPARACION_ASR = pd.DataFrame([
    {
        'persona': PERSONA, 'escenario': audio['id'],
        'completado_con_audio': audio['completado'],
        'completado_con_transcripcion_correcta': replay_por_id.get(audio['id'], {}).get('completado'),
        'resultado_distinto': (
            audio['completado'] != replay_por_id[audio['id']]['completado']
            if audio['id'] in replay_por_id else None
        ),
    }
    for audio in RES_VOZ
])
display(COMPARACION_ASR)


,persona,escenario,completado_con_audio,completado_con_transcripcion_correcta,resultado_distinto
0,diego,E01,True,None,None
1,diego,E02,True,None,None
2,diego,E03,True,None,None
3,diego,E04,True,None,None
4,diego,E05,True,None,None
5,diego,E06,True,None,None


## 8. Latencia y calidad TTS opcional


In [8]:
LATENCIAS = tabla_latencias(RES_VOZ)
if len(LATENCIAS):
    display(LATENCIAS.groupby('etapa').segundos.agg(['mean', 'median', 'max']).sort_values('mean', ascending=False))
print('Cargas iniciales y calidad de voz integradas en el informe final.')

TTS_METRICAS = pd.DataFrame()
if EVALUAR_TTS:
    SEMEJANZA = Semejanza(C.MODELO_SEMEJANZA, 'cpu')
    TTS_METRICAS = tabla_calidad_tts(
        RES_VOZ, RUTA_VOZ_RESULTADOS, ASR, SEMEJANZA, VOZ['referencia'],
    )
    display(TTS_METRICAS)
else:
    print('Calidad TTS incluida en el flujo de evaluación de voz.')


,mean,median,max
etapa,,,
total,18.541381,12.470375,39.857585
asr,8.910973,8.209834,16.276220
recuperacion,5.214650,0.000000,18.550210
llm,4.357143,2.034027,15.970864
herramientas,0.058616,0.020297,0.266253


Cargas iniciales y calidad de voz integradas en el informe final.


## 9. Análisis de aciertos y fallos


In [9]:
ANALISIS = tabla_aciertos_fallos(TODOS, limite=5)
display(ANALISIS[['tipo', 'escenario', 'modo', 'diagnostico_observado']])
if not (~MATRIZ['completado']).any():
    print('No hay fallos en la ejecución consolidada; la tabla conserva hasta cinco aciertos representativos.')


,tipo,escenario,modo,diagnostico_observado
0,acierto,T07,texto,El estado final pasó las condiciones definidas...
1,acierto,T08,texto,El estado final pasó las condiciones definidas...
2,acierto,T09,texto,El estado final pasó las condiciones definidas...
3,acierto,T10,texto,El estado final pasó las condiciones definidas...
4,acierto,T11,texto,El estado final pasó las condiciones definidas...


No hay fallos en la ejecución consolidada; la tabla conserva hasta cinco aciertos representativos.


## 10. Guardado reproducible del informe (opcional)


In [10]:
RUTA_RESULTADOS_EJECUCION = None
if GUARDAR_INFORME:
    RUTA_RESULTADOS_EJECUCION = C.RUTA_RESULTADOS / PERSONA / datetime.now().strftime('%Y%m%d_%H%M%S')
    RUTA_RESULTADOS_EJECUCION.mkdir(parents=True, exist_ok=True)
    MATRIZ.to_csv(RUTA_RESULTADOS_EJECUCION / 'metricas_escenarios.csv', index=False)
    ASR_METRICAS.to_csv(RUTA_RESULTADOS_EJECUCION / 'metricas_asr.csv', index=False)
    LATENCIAS.to_csv(RUTA_RESULTADOS_EJECUCION / 'latencias_por_turno.csv', index=False)
    COMPARACION_ASR.to_csv(RUTA_RESULTADOS_EJECUCION / 'audio_vs_transcripcion.csv', index=False)
    ANALISIS.to_csv(RUTA_RESULTADOS_EJECUCION / 'analisis_aciertos_fallos.csv', index=False)
    if len(TTS_METRICAS):
        TTS_METRICAS.to_csv(RUTA_RESULTADOS_EJECUCION / 'metricas_tts.csv', index=False)
    paquetes = ['torch', 'torchaudio', 'transformers', 'sentence-transformers', 'TTS', 'pandas', 'pyarrow', 'rank-bm25']
    versiones = {}
    for nombre in paquetes:
        try:
            versiones[nombre] = importlib.metadata.version(nombre)
        except importlib.metadata.PackageNotFoundError:
            versiones[nombre] = 'no instalado'
    (RUTA_RESULTADOS_EJECUCION / 'versiones.json').write_text(json.dumps(versiones, indent=2), encoding='utf-8')
    evidencia = {
        'persona': PERSONA, 'voz_asistente': VOICE_ID,
        'configuracion': {
            'ejecutar_texto': EJECUTAR_TEXTO, 'ejecutar_voz': EJECUTAR_VOZ,
            'sintetizar_tts': SINTETIZAR_TTS, 'evaluar_tts': EVALUAR_TTS,
        },
        'resumen': RESUMEN, 'resultados_texto': RES_TEXTO, 'resultados_voz': RES_VOZ,
    }
    (RUTA_RESULTADOS_EJECUCION / 'evidencia.json').write_text(
        json.dumps(evidencia, ensure_ascii=False, indent=2, default=str), encoding='utf-8',
    )
    print('Informe guardado en:', RUTA_RESULTADOS_EJECUCION)
else:
    print('Guardado desactivado. Activa GUARDAR_INFORME=True para crear una copia fechada.')


Informe guardado en: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/output/resultados/diego/20261125_234024


## Conclusión

La matriz consolidada es la fuente principal del informe. Un escenario se considera aprobado solo cuando cumple su estado esperado y no registra incumplimientos de presupuesto, stock, consentimiento o duplicación. Los resultados históricos de otras personas no se mezclan con la persona seleccionada.
